In [3]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import h5py
import re

root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/refining/data_slash_results/"
filename = root + 'archs4_gene_human_v2.1_normalized_LW20230126.h5'
with h5py.File(filename, 'r') as f:
    raw_matrix = f['data/raw_matrix'][:]
    gene_names = f['meta/genes/all_gene_names'][:]
    category_names = f['meta/biosamples/category'][:]

category_names = [x.decode('UTF-8') for x in category_names]
gene_names = [x.decode('UTF-8') for x in gene_names]


category_list  = [ "Heart", "Atrium", "Cardiac Muscle Fiber", "Pericardium", "Valve", "Ventricle","HUVEC", 
    "Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
    "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF",
    "HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
    "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
    "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
    "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus","Granulocyte", "Neutrophil", "Blymphocyte", 
    "Plasma Cell", "Plasmacytoid Dendritic Cell", 
    "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
    "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
    "Thymocyte", "Spleen",
    "Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
    "A431", "HACAT", "HNSCC","Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
    "Vascular Smooth Muscle", "LHCNM2","Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
    "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
    "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron","Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
    "Trachea","Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
    "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
    "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]


category_dict  = {
    "Cardiovascular System" : ["Heart", "Atrium", "Cardiac Muscle Fiber", 
                               "Pericardium", "Valve", "Ventricle",
                               "HUVEC"], 
    
    "Connective Tissue" : ["Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
                           "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF"],

    "Digestive System": ["HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
                         "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
                         "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
                         "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus"],
    
    "Immune System" : ["Granulocyte", "Neutrophil", "Blymphocyte", "Plasma Cell", "Plasmacytoid Dendritic Cell", 
                       "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
                       "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
                       "Thymocyte", "Spleen"],
    
    "Integumentary System" : ["Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
                              "A431", "HACAT", "HNSCC"], 
    
    "Muscular System" : ["Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
                         "Vascular Smooth Muscle", "LHCNM2"], 
    
    "Nervous System" : ["Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
                        "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
                        "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron"], 
    
    "Respiratory System" : ["Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
                            "Trachea"], 
    
    "Urogenital System" : ["Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
                           "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
                           "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]
}

# Convert category_dict value list elements to lowercase
for category, value_list in category_dict.items():
    for i in range(len(value_list)):
        value_list[i] = value_list[i].lower()

In [2]:
# Get category-sample indexes
all_cat_idxs = []
for element in category_list:
    indexes = []
    for i, name in enumerate(category_names):
        if element.lower() in name.lower():
            indexes.append(i)
    all_cat_idxs.append(indexes)

# Initialize the dictionaries to store the averages
gene_tier1cat_avgs = []

for gene_row in raw_matrix:
    values = [[gene_row[i] for i in sub_list] for sub_list in all_cat_idxs]
    means = []
    for value in values:
        if len(value) == 0:
            means.append(np.nan)
        else:
            mean = sum(value)/len(value)
            means.append(mean)
    gene_tier1cat_avgs.append(means)


In [3]:
from IPython.display import display

df = pd.DataFrame(gene_tier1cat_avgs, index=gene_names)
df.columns = category_list
#display(df)

In [4]:
# Save matrix to h5py file
with h5py.File(filename, 'a') as f:
    # Check keys before creation
    print(f['data'].keys())
    
    # Create dataset for matrix
    dataset = f.create_dataset("data/lowest_tier_category_matrix_raw", 
                               data=gene_tier1cat_avgs,
                               dtype=np.float64)
    
    # Check keys after creation
    print(f['data'].keys())

<KeysViewHDF5 ['corrected_matrix', 'highest_tier_category_matrix', 'highest_tier_category_matrix2', 'lowest_tier_category_matrix', 'lowest_tier_category_matrix2', 'qn_matrix', 'raw_matrix', 'threshold_matrix']>
<KeysViewHDF5 ['corrected_matrix', 'highest_tier_category_matrix', 'highest_tier_category_matrix2', 'lowest_tier_category_matrix', 'lowest_tier_category_matrix2', 'lowest_tier_category_matrix_raw', 'qn_matrix', 'raw_matrix', 'threshold_matrix']>


---
# H5PY Check & Clean-up

In [5]:
print("Matrix")
print("Cols:", len(qn_matrix))
print("Rows:", len(qn_matrix[0]))

print("Names Array")
print("Gene Len", len(gene_names))
print("Category Len", len(category_names))

Matrix


NameError: name 'qn_matrix' is not defined

In [ ]:
print("Genes:", len(gene_tier1cat_avgs))
print("Lowest Cat:", len(gene_tier1cat_avgs[0]))

---
# System

In [4]:
# Convert category_dict value list elements to lowercase
for category, value_list in category_dict.items():
    for i in range(len(value_list)):
        value_list[i] = value_list[i].lower()

gene_system_idx = {}
for category, keywords in category_dict.items():
    category_indexes = []
    
    for i, sample_name in enumerate(category_names):
        for keyword in keywords:
            if keyword.lower() in sample_name.lower():
                category_indexes.append(i)
    gene_system_idx[category] = category_indexes

gene_cat_avgs = []
for gene_row in raw_matrix:
    cat_avg = []

    for system, indexes in gene_system_idx.items():
        values = [gene_row[index] for index in indexes]
        mean = sum(values)/len(values)
        cat_avg.append(mean)
    gene_cat_avgs.append(cat_avg)
    


In [5]:
print("Genes:", len(gene_cat_avgs))
print("Highest Cat:", len(gene_cat_avgs[0]))

Genes: 57447
Highest Cat: 9


In [10]:
gene_cat_avgs[0][0:3]

[94.79237844940867, 99.22251054168018, 113.55603685910057]

In [8]:
# Save matrix to h5py file
with h5py.File(filename, 'a') as f:
    # Check keys before creation
    print(f['data'].keys())
    
    # Create dataset for matrix
    dataset = f.create_dataset("data/highest_tier_category_matrix_raw", 
                               data=gene_cat_avgs,
                               dtype=np.float64)
    
    # Check keys after creation
    print(f['data'].keys())

<KeysViewHDF5 ['corrected_matrix', 'highest_tier_category_matrix', 'highest_tier_category_matrix2', 'lowest_tier_category_matrix', 'lowest_tier_category_matrix2', 'lowest_tier_category_matrix_raw', 'qn_matrix', 'raw_matrix', 'threshold_matrix']>
<KeysViewHDF5 ['corrected_matrix', 'highest_tier_category_matrix', 'highest_tier_category_matrix2', 'highest_tier_category_matrix_raw', 'lowest_tier_category_matrix', 'lowest_tier_category_matrix2', 'lowest_tier_category_matrix_raw', 'qn_matrix', 'raw_matrix', 'threshold_matrix']>


In [7]:
# import h5py

# # Open the HDF5 file in read/write mode
# with h5py.File(filename, 'r+') as f:
    
#     # Get a reference to the "data" group
#     data_group = f['data']
    
#     # Delete the "highest_tier_category_matrix_raw" dataset
#     del data_group['highest_tier_category_matrix_raw']
    
#     # Close the file
#     f.close()
